
# BigQuery Fundamentals — Indian Employee Dataset Hands-On Lab
### GCP Training Program — Day 1: Storage & Ingestion (BigQuery Basics)

## Problem Statement

A small Indian technology company ("IndexCorp") wants to move its HR data into BigQuery to answer
basic workforce questions — headcount by department, salary distribution by gender and state,
attendance trends over time — and eventually wants those queries to run fast and cheaply even as
the attendance history grows to millions of rows. This notebook builds up the exact BigQuery
techniques needed to do that, one concept at a time: loading data, basic querying, aggregation,
joins, views, materialized views, partitioning, and clustering.

## The Dataset

Three CSV files, hosted on GitHub, designed to work together as a small but realistic HR data
model — one dimension table for departments, one dimension/fact hybrid for employees, and one
large fact table for monthly attendance (this is what makes the partitioning and clustering
sections meaningful rather than abstract).

**Source:** `https://github.com/himanshurathi/gcp-training-labs-accordion/tree/master/day-1-foundations`

### `departments.csv` — 8 rows (dimension table)
One row per department. Small, slowly-changing — a classic dimension table.

| Column | Type | Description |
|---|---|---|
| `department_id` | INTEGER | Unique department identifier (101-108) |
| `department_name` | STRING | e.g. Engineering, Sales, Finance |
| `location` | STRING | City where the department is based |
| `department_head` | STRING | Name of the department's head |

### `employees.csv` — 100 rows (dimension/fact hybrid)
One row per employee. The main table most basic queries and joins will use.

| Column | Type | Description |
|---|---|---|
| `employee_id` | STRING | Unique ID, e.g. EMP1000 |
| `first_name` / `last_name` | STRING | Employee's name |
| `gender` | STRING | Male / Female |
| `state` / `city` | STRING | Indian state of residence and corresponding city |
| `department_id` | INTEGER | Foreign key to `departments.department_id` |
| `designation` | STRING | Job title, e.g. Software Engineer, Manager |
| `date_of_joining` | DATE | YYYY-MM-DD |
| `monthly_salary_inr` | FLOAT | Monthly salary in Indian Rupees |
| `email` | STRING | Employee's email address |

### `attendance_monthly.csv` — 1,200 rows (large fact table)
One row per employee, per month, for all of 2025 (100 employees × 12 months). This is the table
we'll use for partitioning and clustering, since it's the only one with a real time dimension and
enough rows to make pruning meaningfully visible.

| Column | Type | Description |
|---|---|---|
| `employee_id` | STRING | Foreign key to `employees.employee_id` |
| `department_id` / `state` | INTEGER / STRING | Denormalized for convenient clustering/filtering |
| `month_date` | DATE | First day of the month, e.g. 2025-03-01 |
| `days_present` / `leaves_taken` | INTEGER | Monthly attendance counts |
| `performance_rating` | FLOAT | Self-reported monthly rating, 1.0-5.0 |

**Why denormalize `department_id` and `state` into `attendance_monthly`?** In a real warehouse
you'd often join back to `employees` for these. We include them directly here so the partitioning
and clustering sections can demonstrate pruning on this table without requiring a join in every
example query — trading a little redundancy for simpler, cheaper downstream queries.

**This notebook is fully self-contained.** Authenticate → Configure → the CSVs are pulled directly
from GitHub, no manual upload needed → everything else runs against your own project.


## 1. Authenticate This Session

In [ ]:

import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab. This also configures gcloud/bq for this session.")
else:
    print("Not running in Colab — assuming gcloud Application Default Credentials are set up.")



## 2. Configure Your Project & Create a Working Dataset

A dataset called `hr_analytics` is created automatically to hold every table, view, and
materialized view this notebook builds.


In [ ]:

PROJECT_ID = input("Enter your GCP Project ID: ").strip()
LOCATION = input("Enter your BigQuery location [default: US]: ").strip() or "US"
DATASET_ID = "hr_analytics"

!gcloud config set project {PROJECT_ID}
!gcloud services enable bigquery.googleapis.com --project {PROJECT_ID}

from google.cloud import bigquery
from google.cloud.exceptions import Conflict
import pandas as pd

bq_client = bigquery.Client(project=PROJECT_ID)

def TBL(name):
    return f"`{PROJECT_ID}.{DATASET_ID}.{name}`"

def run_query(sql, label=None, expect_rows=True):
    if label:
        print(f"--- {label} ---")
    job = bq_client.query(sql)
    if expect_rows:
        try:
            df = job.result().to_dataframe()
            display(df)
        except Exception:
            job.result()
    else:
        job.result()
    if job.total_bytes_processed is not None:
        print(f"Bytes processed: {job.total_bytes_processed:,}")
    return job

dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = LOCATION
try:
    bq_client.create_dataset(dataset_ref)
    print(f"Created dataset {DATASET_ID}")
except Conflict:
    print(f"Dataset {DATASET_ID} already exists — continuing.")

print(f"\nProject:  {PROJECT_ID}")
print(f"Location: {LOCATION}")
print(f"Dataset:  {DATASET_ID}")



> Check it in the console: **BigQuery > SQL workspace** → your project → the new
> `hr_analytics` dataset should now appear in the Explorer panel, empty for now.



## 3. Loading the CSVs Into BigQuery — Directly From GitHub

**What this does:** rather than a manual upload step, we pull each CSV straight from its GitHub
raw URL with `pandas.read_csv`, then load it into BigQuery with `load_table_from_dataframe` —
schema inference happens automatically from the DataFrame's dtypes, the same convenience
`--autodetect` gives the `bq load` CLI.

**On schema inference:** convenient for small, clean files like these, but in a production
pipeline it's safer to declare an explicit schema so a single malformed row can't silently change
a column's inferred type.


In [ ]:

GITHUB_BASE = (
    "https://raw.githubusercontent.com/himanshurathi/"
    "gcp-training-labs-accordion/master/day-1-foundations"
)

departments_df = pd.read_csv(f"{GITHUB_BASE}/departments.csv")
employees_df = pd.read_csv(f"{GITHUB_BASE}/employees.csv")
attendance_df = pd.read_csv(f"{GITHUB_BASE}/attendance_monthly.csv")

print("departments:", departments_df.shape)
print("employees:", employees_df.shape)
print("attendance_monthly:", attendance_df.shape)
departments_df.head(3)


In [ ]:

load_config = bigquery.LoadJobConfig(write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE)

bq_client.load_table_from_dataframe(departments_df, TBL("departments").strip("`"), job_config=load_config).result()
print("departments loaded:", bq_client.get_table(TBL("departments").strip("`")).num_rows, "rows")

bq_client.load_table_from_dataframe(employees_df, TBL("employees").strip("`"), job_config=load_config).result()
print("employees loaded:", bq_client.get_table(TBL("employees").strip("`")).num_rows, "rows")

bq_client.load_table_from_dataframe(attendance_df, TBL("attendance_monthly").strip("`"), job_config=load_config).result()
print("attendance_monthly loaded:", bq_client.get_table(TBL("attendance_monthly").strip("`")).num_rows, "rows")



> Check it in the console: **BigQuery > SQL workspace > hr_analytics** → all three tables
> should now be listed. Click any one → the **Schema** tab shows exactly what was inferred from
> the DataFrame, and the **Preview** tab shows the loaded rows without running a query.



## 4. Basic Queries — SELECT, WHERE, ORDER BY

**The question we're answering:** "Show me all employees in Karnataka earning over ₹80,000 a
month, highest paid first." The simplest possible shape of query: filter rows, sort them, look at
the result.


In [ ]:
run_query(f"""
SELECT
  employee_id, first_name, last_name, state, designation, monthly_salary_inr
FROM {TBL("employees")}
WHERE state = 'Karnataka' AND monthly_salary_inr > 80000
ORDER BY monthly_salary_inr DESC
""", label="Employees in Karnataka earning over 80,000/month")



> Check it in the console: paste this into the BigQuery SQL workspace and click Run. The
> **Job information** panel below the results grid shows **Bytes processed** — on a 100-row table
> this will be tiny (a few KB), worth contrasting later with the 1,200-row attendance table.

### 4.1 Aggregate Functions — COUNT, AVG, SUM
**The question:** "What's the average salary company-wide, and how many employees do we have?"


In [ ]:
run_query(f"""
SELECT
  COUNT(*) AS total_employees,
  ROUND(AVG(monthly_salary_inr), 2) AS avg_salary,
  MIN(monthly_salary_inr) AS min_salary,
  MAX(monthly_salary_inr) AS max_salary
FROM {TBL("employees")}
""", label="Company-wide salary stats")



### 4.2 GROUP BY — Headcount and Average Salary by State
**The question:** "Break that down by state." `GROUP BY` collapses all rows sharing the same
state into a single output row, with the aggregate functions computed per group.


In [ ]:
run_query(f"""
SELECT
  state,
  COUNT(*) AS headcount,
  ROUND(AVG(monthly_salary_inr), 2) AS avg_salary
FROM {TBL("employees")}
GROUP BY state
ORDER BY headcount DESC
""", label="Headcount and avg salary by state")



> Check it in the console: run this, then click any column header in the results grid to
> sort interactively without re-running the query — a quick way to explore a small result set
> live.

### 4.3 A Window Function — Salary Comparison Within Each Department
**The question:** "For each employee, how does their salary compare to their department's
average?" A window function (`AVG(...) OVER (PARTITION BY ...)`) computes an aggregate *per
group* while still returning one row per employee — unlike `GROUP BY`, which collapses rows.


In [ ]:
run_query(f"""
SELECT
  e.employee_id, e.first_name, e.department_id, e.monthly_salary_inr,
  ROUND(AVG(e.monthly_salary_inr) OVER (PARTITION BY e.department_id), 2)
    AS dept_avg_salary,
  ROUND(e.monthly_salary_inr -
    AVG(e.monthly_salary_inr) OVER (PARTITION BY e.department_id), 2)
    AS diff_from_dept_avg
FROM {TBL("employees")} AS e
ORDER BY e.department_id, diff_from_dept_avg DESC
""", label="Salary vs. department average")



**`GROUP BY` vs. window functions:** `GROUP BY` answers "what's the average per department" (one
row per department). A window function answers "what's the average per department, shown
alongside every individual employee row" (one row per employee, still). Both are useful; the
choice depends on whether you need to keep the individual rows visible.



## 5. Joins

**The question we're answering:** "Show me employee names alongside their department name and
location — not just a `department_id` number." `employees` only has `department_id`; the readable
`department_name` lives in `departments`. A join combines rows from both tables based on a
matching key.

### 5.1 INNER JOIN — Employees With Department Details


In [ ]:
run_query(f"""
SELECT
  e.employee_id, e.first_name, e.last_name,
  d.department_name, d.location
FROM {TBL("employees")} AS e
INNER JOIN {TBL("departments")} AS d
  ON e.department_id = d.department_id
LIMIT 20
""", label="Employees with department details")



**`INNER JOIN` keeps only matches:** if an employee's `department_id` didn't exist in the
departments table (e.g. a typo, or a department that was since deleted), that employee would
simply not appear in the result at all — silently dropped. Section 5.3 shows how to catch exactly
this.

### 5.2 Aggregation Across a Join — Average Salary per Department
**The question:** "Which departments pay the most on average, and how many people are in each?"
This combines a join with `GROUP BY` — a very common real-world pattern.


In [ ]:
run_query(f"""
SELECT
  d.department_name, d.location,
  COUNT(e.employee_id) AS headcount,
  ROUND(AVG(e.monthly_salary_inr), 2) AS avg_salary
FROM {TBL("departments")} AS d
INNER JOIN {TBL("employees")} AS e
  ON d.department_id = e.department_id
GROUP BY d.department_name, d.location
ORDER BY avg_salary DESC
""", label="Average salary per department")



### 5.3 LEFT JOIN — Find Departments With Zero Employees
**The question:** "Do we have any departments with nobody assigned to them?" A `LEFT JOIN` keeps
every row from the left table (departments) regardless of whether a match exists on the right
(employees) — unmatched rows get `NULL` for every employees column. Filtering
`WHERE e.employee_id IS NULL` isolates exactly the departments with no staff.


In [ ]:
run_query(f"""
SELECT
  d.department_id, d.department_name
FROM {TBL("departments")} AS d
LEFT JOIN {TBL("employees")} AS e
  ON d.department_id = e.department_id
WHERE e.employee_id IS NULL
""", label="Departments with zero employees")



> Check it in the console: run each of these three in the SQL workspace. For the
> `LEFT JOIN`, try changing it back to `INNER JOIN` and re-running — the departments-with-no-staff
> row (if any) will silently disappear, a concrete demonstration of the exact difference between
> the two join types.



## 6. Views

**The problem this solves:** the department-summary join from Section 5.2 is useful, but
re-typing that `JOIN` + `GROUP BY` every time an analyst wants this information is repetitive and
error-prone. A `VIEW` stores the *query itself*, not its results — querying the view re-runs the
underlying `SELECT` against the live base tables every time.


In [ ]:
run_query(f"""
CREATE OR REPLACE VIEW {TBL("department_summary_view")} AS
SELECT
  d.department_name, d.location,
  COUNT(e.employee_id) AS headcount,
  ROUND(AVG(e.monthly_salary_inr), 2) AS avg_salary
FROM {TBL("departments")} AS d
INNER JOIN {TBL("employees")} AS e
  ON d.department_id = e.department_id
GROUP BY d.department_name, d.location
""", label="Create view", expect_rows=False)

run_query(f"""
SELECT * FROM {TBL("department_summary_view")}
ORDER BY avg_salary DESC
""", label="Query the view like a table")



> Check it in the console: **hr_analytics** dataset → `department_summary_view` shows a
> small "view" icon distinguishing it from a table → its **Details** tab shows **Table type:
> VIEW** and the exact SQL stored — confirming a view holds no data of its own, only a query
> definition.

**Advantage vs. trade-off:** a view is always 100% live (it can never show stale data) and costs
nothing to store. But it gives *no performance benefit* — every query against it re-runs the full
underlying join and aggregation from scratch, exactly as expensive as writing the raw SQL
yourself. That's the problem Section 7 (Materialized Views) solves.



## 7. Materialized Views

**The problem this solves:** if `department_summary_view` were queried constantly by a live
dashboard, re-computing the join and aggregation on every single refresh is wasteful. A
`MATERIALIZED VIEW` actually **precomputes and stores** the result, then keeps it incrementally
refreshed as the base tables change.


In [ ]:
run_query(f"""
CREATE MATERIALIZED VIEW {TBL("monthly_attendance_summary_mv")} AS
SELECT
  a.month_date,
  a.department_id,
  ROUND(AVG(a.days_present), 2) AS avg_days_present,
  ROUND(AVG(a.performance_rating), 2) AS avg_performance_rating,
  COUNT(*) AS employee_month_records
FROM {TBL("attendance_monthly")} AS a
GROUP BY a.month_date, a.department_id
""", label="Create materialized view", expect_rows=False)

run_query(f"""
SELECT *
FROM {TBL("monthly_attendance_summary_mv")}
WHERE month_date = '2025-06-01'
ORDER BY avg_performance_rating DESC
""", label="Query it exactly like a table")



> Check it in the console: **hr_analytics** dataset → `monthly_attendance_summary_mv`
> should show a distinct "materialized view" icon. Its **Details** tab has a **Refresh** section
> (last refresh time, staleness) that a plain view's Details tab never shows, since a plain view
> has no data to refresh in the first place.

**Limitations to mention:** materialized views support a restricted SQL subset (no
non-deterministic functions like `CURRENT_TIMESTAMP()`, limited join types) — check current
BigQuery docs for the exact supported surface. They also cost storage, unlike a plain view, and
refresh has some latency (near-real-time, not instantaneous).



## 8. Partitioning

**The problem this solves:** `attendance_monthly` has 1,200 rows in this small demo — trivial to
scan in full every time. But imagine this same table after 5 years across 10,000 employees:
millions of rows. A query asking about just June 2025 shouldn't have to scan every other month
too. `PARTITION BY` splits a table into physical chunks by a date column, so a query filtering on
that column can skip irrelevant chunks entirely.


In [ ]:
run_query(f"""
CREATE TABLE {TBL("attendance_monthly_partitioned")}
PARTITION BY month_date AS
SELECT *
FROM {TBL("attendance_monthly")}
""", label="Create partitioned copy", expect_rows=False)



**Seeing the benefit:** compare the estimated bytes scanned for the same filter, against the
unpartitioned table versus the partitioned copy, using a dry run.


In [ ]:
unpartitioned_job = bq_client.query(
    f'SELECT AVG(days_present) FROM {TBL("attendance_monthly")} WHERE month_date = "2025-06-01"',
    job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False),
)
partitioned_job = bq_client.query(
    f'SELECT AVG(days_present) FROM {TBL("attendance_monthly_partitioned")} WHERE month_date = "2025-06-01"',
    job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False),
)

print(f"Unpartitioned table — estimated bytes: {unpartitioned_job.total_bytes_processed:,}")
print(f"Partitioned table    — estimated bytes: {partitioned_job.total_bytes_processed:,}")



> Check it in the console: **hr_analytics > attendance_monthly_partitioned** → **Details**
> tab → a **Partition** section confirms partitioning by `month_date`, and **Table storage** shows
> per-partition row counts — visual proof the table really is split into monthly chunks, not just
> labeled as one.

**At this table's small size, the byte-count difference will be modest** — the real-world payoff
of partitioning only becomes dramatic at the millions-of-rows scale this demo dataset is
deliberately too small to simulate. Worth saying explicitly to a class: *the mechanism being
demonstrated here is exactly what saves enormous cost at real production scale, even though the
numbers on screen today look unremarkable.*



## 9. Clustering

**The problem this solves:** partitioning narrows a query down to one month. But within that
month, if we only care about one department, we're still scanning every department's rows for
that month. `CLUSTER BY` sorts data *within* each partition by the given column(s) — a query
filtering on that column can then skip blocks of data within a partition too.


In [ ]:
run_query(f"""
CREATE TABLE {TBL("attendance_monthly_partitioned_clustered")}
PARTITION BY month_date
CLUSTER BY department_id AS
SELECT *
FROM {TBL("attendance_monthly")}
""", label="Create partitioned + clustered copy", expect_rows=False)


In [ ]:
run_query(f"""
SELECT AVG(performance_rating)
FROM {TBL("attendance_monthly_partitioned")}
WHERE month_date = '2025-06-01' AND department_id = 101
""", label="Partitioned-only, filtered by month + department")

run_query(f"""
SELECT AVG(performance_rating)
FROM {TBL("attendance_monthly_partitioned_clustered")}
WHERE month_date = '2025-06-01' AND department_id = 101
""", label="Partitioned + clustered, same filter")



> Check it in the console: **attendance_monthly_partitioned_clustered** → **Details** tab
> shows both a **Partition** section AND a **Clustering** section listing `department_id` — the
> one place in the console where both optimizations are visible on the same table at once.

**Caveat:** clustering's benefit doesn't always show up clearly in a dry-run estimate the way
partition pruning does — for a ground-truth comparison, run both queries for real and compare
actual **Bytes billed** in the completed job's details, not just the dry-run estimate.



## 10. Cleanup

Once the demonstration is complete, remove everything created in this notebook — the
materialized view, the view, both partitioned/clustered table copies, the three base tables, and
the dataset itself. Guarded by `CONFIRM_DELETE`.


In [ ]:

CONFIRM_DELETE = False  # <-- set to True to actually delete resources

if not CONFIRM_DELETE:
    print("CONFIRM_DELETE is False — nothing will be deleted. "
          "Set CONFIRM_DELETE = True above and re-run this cell when you're ready.")


In [ ]:

if CONFIRM_DELETE:
    try:
        bq_client.delete_dataset(
            f"{PROJECT_ID}.{DATASET_ID}", delete_contents=True, not_found_ok=True
        )
        print(f"Deleted dataset {DATASET_ID} (every table, view, and materialized view inside it).")
    except Exception as e:
        print("Cleanup skipped/failed:", e)

    print("\nCleanup complete.")



> Final console check: **BigQuery > SQL workspace > Explorer** → the `hr_analytics`
> dataset should no longer be listed under your project at all, confirming every table, view, and
> materialized view created in this notebook was removed along with it.

## Companion Reference
This notebook's concept sequence — queries → joins → views → materialized views → partitioning →
clustering — matches the standalone PDF hand-guide covering the same material with this same
dataset, useful as a print-friendly leave-behind reference or for a session where a live Colab
isn't practical.
